In [ ]:
# Импорты

from pathlib import Path
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from tqdm.auto import tqdm
import json

d:\DataScience\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import os
import sys
from pathlib import Path
sys.path.append(os.path.abspath(".."))

In [ ]:
from src.embeddings import load_resnet_model
from src.embeddings import get_resnet_embedding
from src.embeddings import generate_resnet_embeddings

In [ ]:
# Пути

ROOT = Path("..")

PROCESSED_PATH = ROOT / "data" / "processed"
MODELS_PATH = ROOT / "models"
EMBEDDINGS_PATH = ROOT / "embeddings" / "resnet50_train"

# Создаем директорию для сохранения эмбеддингов
EMBEDDINGS_PATH.mkdir(parents=True, exist_ok=True)

# Проверяем используемые пути
print(f"Processed:  {PROCESSED_PATH}")
print(f"Models:    {MODELS_PATH}")
print(f"Embeddings: {EMBEDDINGS_PATH}")

Processed:  ..\data\processed
Models:    ..\models
Embeddings: ..\embeddings\resnet50_train


In [ ]:
SPLITS_PATH = ROOT / "data" / "splits"

with open(SPLITS_PATH / "train.json", "r", encoding="utf-8") as f:
    train_images = json.load(f)

print(f"Train images: {len(train_images)}")

Train images: 16806


In [ ]:
# Классы и устройство

# Получаем список классов из структуры датасета
CLASSES = sorted(
    folder.name
    for folder in PROCESSED_PATH.iterdir()
    if folder.is_dir()
)

# Определяем устройство для вычислений
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Classes: {len(CLASSES)}")
print(f"Device: {device}")

Classes: 23
Device: cuda


In [ ]:
resnet50 = load_resnet_model(
    models_path=MODELS_PATH,
    device=device
)

d:\DataScience\My_Projects\anime_project(final_project)\src\embeddings.py:199: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(


Classification ResNet loaded.


In [ ]:
# Преобразования изображений

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
embedding = get_resnet_embedding(
    image_path=train_images[0]["path"],
    model=resnet50,
    transform=transform,
    device=device
)

print(embedding.shape)

(2048,)


In [ ]:
SAVE_STEP = 100

embeddings, labels, paths = generate_resnet_embeddings(
    train_images=train_images,
    save_dir=EMBEDDINGS_PATH,
    model=resnet50,
    transform=transform,
    device=device,
    save_step=SAVE_STEP,
)

Продолжаем: 16806


Embedding train images: 100%|██████████| 16806/16806 [00:00<00:00, 271812.75it/s]



Готово: 16806


In [ ]:
# Проверка сохраненных данных

print(f"Embeddings shape: {embeddings.shape}")
print(f"Labels shape:     {labels.shape}")
print(f"Paths shape:      {paths.shape}")

print()

print(f"Embedding size:   {embeddings.shape[1]}")
print(f"Embedding dtype:  {embeddings.dtype}")

print()

print(f"First label:      {labels[0]}")
print(f"First path:       {paths[0]}")

Embeddings shape: (16806, 2048)
Labels shape:     (16806,)
Paths shape:      (16806,)

Embedding size:   2048
Embedding dtype:  float32

First label:      One_Punch_Man
First path:       ..\data\processed\One_Punch_Man\_720_964579835_00244.jpg
